In [ ]:
# MT Adaptation with SmolDoc: SFT on H100

import os
import torch

from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer

from trl import SFTTrainer, SFTConfig

from utils import get_extended_datasets

torch.set_float32_matmul_precision("high")  # helps on H100

# --- CONFIG ---

SMOLDOC_CONFIG = "smoldoc__en_sw"

# Base model for full SFT (change this to other models: llama, gemma, etc.)
SFT_MODEL_NAME = "google/gemma-3-4b-pt"

OUTPUT_DIR_SFT = f"checkpoints/sft_{SMOLDOC_CONFIG}"
os.makedirs("checkpoints", exist_ok=True)

print("Config:", SMOLDOC_CONFIG)
print("SFT base model:", SFT_MODEL_NAME)

# --- Load SmolDoc + factuality annotations, pick one config, build text field ---

datasets = get_extended_datasets(save_path="data/smoldoc_datasets", overwrite=False)
ds_full: Dataset = datasets[SMOLDOC_CONFIG]

print(ds_full)

# Simple train/validation split
mt_ds: DatasetDict = ds_full.train_test_split(test_size=0.1, seed=42)
train_ds = mt_ds["train"]
eval_ds = mt_ds["test"]

print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))


def format_mt_example(srcs, trgs, lang_name="Swahili"):
    """
    Build a single text sequence of the form:

    You are an expert in English to Swahili translation.
    Translate the following English text into Swahili.

    English:
    <src>

    Swahili:
    <tgt>
    """
    src = " ".join(srcs).strip()
    tgt = " ".join(trgs).strip()
    return (
        "You are an expert in English to Swahili translation.\n"
        "Translate the following English text into Swahili.\n\n"
        f"English:\n{src}\n\nSwahili:\n{tgt}"
    )


def add_text_column(batch):
    texts = [
        format_mt_example(srcs, trgs, lang_name="Swahili")
        for srcs, trgs in zip(batch["srcs"], batch["trgs"])
    ]
    return {"text": texts}

def formatting_func(examples):
    # TRL passes a batch as a dict of lists, e.g. {"text": [...], "id": [...]}
    # We just return the text list.
    return examples["text"]

train_ds_fmt = train_ds.map(add_text_column, batched=True)
eval_ds_fmt = eval_ds.map(add_text_column, batched=True)

print("Example training text:\n", train_ds_fmt[0]["text"][:400])

In [ ]:
import evaluate
import numpy as np
from transformers import EvalPrediction

# Load the BLEU metric
bleu_metric = evaluate.load("bleu")

def postprocess_text(preds, labels):
    # This function is crucial for cleaning the generated text for BLEU calculation.
    # The format is: "...Swahili:\n<tgt>"

    # 1. Clean up predictions
    extracted_preds = []
    for pred in preds:
        # Isolate the text that comes *after* "Swahili:"
        try:
            translation = pred.split("Swahili:")[-1].strip()
            # Stop decoding after the first line break or EOS, if applicable
            translation = translation.split('\n')[0].strip()
            extracted_preds.append(translation)
        except:
            extracted_preds.append("") # Handle unexpected formatting

    # 2. Clean up labels
    # The labels are the full formatted text, so we extract the true target (<tgt>)
    extracted_labels = []
    for label in labels:
        try:
            target = label.split("Swahili:")[-1].strip()
            target = target.split('\n')[0].strip()
            extracted_labels.append([target]) # BLEU expects a list of references, even if only one
        except:
            extracted_labels.append([""])

    return extracted_preds, extracted_labels

def compute_metrics_mt(p: EvalPrediction):
    # p.predictions are the token IDs generated by the model's forward pass.
    # For a CausalLM (like Gemma) in SFTTrainer, we use `argmax` to get the predicted token IDs.

    # 1. Get predicted token IDs (assuming no `predict_with_generate=True` for CausalLM)
    if isinstance(p.predictions, tuple):
        # Handle cases where the model returns logits and other outputs (e.g., past_key_values)
        preds = p.predictions[0]
    else:
        preds = p.predictions

    # Take the argmax over the vocabulary dimension to get the most likely token ID
    # If `preprocess_logits_for_metrics` is used, `preds` are already token IDs
    # with shape (batch, seq_len). If not, they are logits with shape
    # (batch, seq_len, vocab_size) and we need an argmax over the vocab dim.
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    labels = p.label_ids

    # Replace the loss mask token (-100) with the tokenizer's padding token ID
    labels = np.where(labels != -100, labels, tokenizer_sft.pad_token_id)

    # 2. Decode the token IDs to text
    decoded_preds = tokenizer_sft.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer_sft.batch_decode(labels, skip_special_tokens=True)

    # 3. Post-process to extract only the translation
    final_preds, final_labels = postprocess_text(decoded_preds, decoded_labels)

    # 4. Compute BLEU score
    # Note: Using BLEU on the decoded loss-prediction sequence is an approximation
    # and might not perfectly reflect the score from full beam-search generation.
    bleu_result = bleu_metric.compute(predictions=final_preds, references=final_labels)

    return {"bleu": bleu_result["bleu"]}

In [ ]:
# --- Supervised Fine-Tuning (SFT) on H100 ---
from sklearn.model_selection import KFold
import copy
NUM_EPOCHS = 3
K_FOLDS = 5
# --- Tokenizer ---
tokenizer_sft = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)
if tokenizer_sft.pad_token is None:
    tokenizer_sft.pad_token = tokenizer_sft.eos_token
tokenizer_sft.padding_side = "right"

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1) # Return the predicted token ID (argmax)
# H100 is big, so push the batch size
# Safe starting point for a 4B model @ 2k tokens on 80GB:
per_device_bs = 1        # bump to 8 if it fits
grad_accum = 4           # so global batch = 4 * 8 = 32 sequences

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR_SFT,
    num_train_epochs=NUM_EPOCHS,                    # increase when you're happy with stability
    per_device_train_batch_size=per_device_bs,
    per_device_eval_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,
    learning_rate=5e-5,
    max_length=1024,                  # NOTE: max_seq_length, not max_length
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=200,
    bf16=True,                            # H100 loves bf16
    packing=False,                        # can try True later for better throughput
    dataloader_num_workers=4,             # use your CPU a bit
    gradient_checkpointing=False,         # you *can* turn this on, but you have VRAM
    optim="adamw_torch_fused",            # good on NVIDIA GPUs
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
)

# --- K-FOLD CROSS-VALIDATION LOOP (K=5) ---

# Set the number of epochs in the config
sft_config.num_train_epochs = NUM_EPOCHS

# Convert the full dataset to a list of indices for K-Fold
all_indices = np.arange(len(ds_full))

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

for fold, (train_index, eval_index) in enumerate(kf.split(all_indices)):
    print(f"\n{'='*50}\nSTARTING FOLD {fold + 1}/{K_FOLDS}\n{'='*50}")

    # 1. Split data for the current fold
    # Get subsets using the indices
    train_ds_fold = ds_full.select(train_index)
    eval_ds_fold = ds_full.select(eval_index)

    # Apply the formatting function
    train_ds_fmt_fold = train_ds_fold.map(add_text_column, batched=True)
    eval_ds_fmt_fold = eval_ds_fold.map(add_text_column, batched=True)

    print(f"Fold {fold + 1} | Train size: {len(train_ds_fmt_fold)} | Eval size: {len(eval_ds_fmt_fold)}")

    # 2. Re-initialize model and tokenizer for each fold
    # This prevents carry-over of weights from the previous fold
    # Note: If memory is a concern, you might only re-initialize the model,
    # but re-initialization is safer for true CV.

    # Model (copy the initial weights or reload from scratch)
    model_sft_fold = AutoModelForCausalLM.from_pretrained(
        SFT_MODEL_NAME,
        dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    model_sft_fold.gradient_checkpointing_enable()
    model_sft_fold.config.use_cache = False

    # 3. Create Trainer and Train
    output_dir_fold = os.path.join(OUTPUT_DIR_SFT, f"fold_{fold + 1}")
    sft_config_fold = copy.deepcopy(sft_config)
    sft_config_fold.output_dir = output_dir_fold

    trainer_sft_fold = SFTTrainer(
        model=model_sft_fold,
        processing_class=tokenizer_sft,
        train_dataset=train_ds_fmt_fold,
        eval_dataset=eval_ds_fmt_fold,
        formatting_func=formatting_func,
        args=sft_config_fold,
        compute_metrics=compute_metrics_mt, # Assuming you added the custom metric from the previous answer
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )

    trainer_sft_fold.train()

    # Save final checkpoint for this fold
    final_dir_fold = os.path.join(output_dir_fold, "final")
    trainer_sft_fold.save_model(final_dir_fold)
    tokenizer_sft.save_pretrained(final_dir_fold)

    print(f"Fold {fold + 1} training finished. Saved to: {final_dir_fold}")

    # Cleanup memory (optional but recommended for H100 memory management)
    del model_sft_fold, trainer_sft_fold
    torch.cuda.empty_cache()

# --- END OF K-FOLD LOOP ---

In [ ]:
from transformers import pipeline

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR_SFT, "final")

ft_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"

ft_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",   # single H100
)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    device_map="cuda:0",
)

# Build a test prompt using the same pattern as training
test_src = "I love machine learning."
prompt = (
    "You are an expert in English to Swahili translation.\n"
    "Translate the following English text into Swahili.\n\n"
    f"English:\n{test_src}\n\nSwahili:\n"
)

out = pipe(
    prompt,
    max_new_tokens=60,
    do_sample=False,  # deterministic sanity
)

print("=== PROMPT ===")
print(prompt)
print("\n=== MODEL COMPLETION ===")
print(out[0]["generated_text"])

In [ ]:
# Use the final model from the last trained fold (adjust CHECKPOINT_DIR path)
# If you want to use the first split's test set for a final check:
test_ds = mt_ds["test"] # The original test split of 59 samples

# Path to the final model (assuming you pick the last fold's result)
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR_SFT, f"fold_{K_FOLDS}", "final")

# --- Load Model and Pipeline ---

ft_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "left" # Set to left for better generation speed/results
# ... (rest of model loading code as you had it) ...

ft_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",   # single H100
)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    device_map="cuda:0",
)

print("\n\n" + "="*50)
print("FINAL TEST SET TRANSLATIONS (Original Test Split)")
print("="*50)

for i in range(len(test_ds)):
    example = test_ds[i]
    src_text = " ".join(example["srcs"]).strip()
    ref_text = " ".join(example["trgs"]).strip()

    # Build the prompt pattern for inference (without the target text)
    prompt = (
        "You are an expert in English to Swahili translation.\n"
        "Translate the following English text into Swahili.\n\n"
        f"English:\n{src_text}\n\nSwahili:\n"
    )

    out = pipe(
        prompt,
        max_new_tokens=100, # Increased max tokens for full translation
        do_sample=False,
        # Stop generation when the model predicts the end of its intended output (like a new line)
        return_full_text=False, # Only return the generated part
    )

    # Extract and clean the generated translation
    generated_text = out[0]["generated_text"].split('\n')[0].strip()

    print(f"\n--- Example {i+1} / {len(test_ds)} ---")
    print(f"SOURCE (EN): {src_text}")
    print(f"REFERENCE (SW): {ref_text}")
    print(f"MODEL OUTPUT (SW): {generated_text}")

# End of inference block